In [1]:
import os

os.chdir('..')
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # set for determinism

## Loading the Dataset:

In this section the pointcloud is loaded. The SIREN paper suggests normalizing the point coordinates as periodic activations implicitly expect a bounded input. 

In [2]:
import open3d as o3d
import numpy as np
import torch
import matplotlib.pyplot as plt
import src.model.SIREN as si
from src.model.training import train
import src.loss.SDF_loss as loss
from src.mesh_extraction.marching_cubes_test import write_obj
import src.model.MLP as simple
import src.data.dataset as data
import src.model.pruning_module as pm
import src.mesh_extraction.marching_cubes_gpu as marching_cubes


np.random.seed(42)
torch.cuda.manual_seed_all(42)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

mesh = data.MeshDataset('data/pointclouds/Armadillo/Armadillo.ply')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Defining the Model

In this cell we will define the SIREN model. This particular INR uses sine activations for nonlinearity and is supposed to capture more information given the underlying data when compared to a model that uses ReLU activations. This way, a good INR accuracy can be achieved with fewer neurons.

In [4]:
size_per_layer = 256
torch.manual_seed(42)
model = si.SIRENSDF()
model_loss = loss.Loss(lambda_surface=150,  lambda_eikonal=0.5, lambda_normal=1.5, lambda_inter=0.5, lambda_sign=1.5, lambda_twd=1e-4, model=model)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
si.weight_stats(model)

Weight statistics per layer:
------------------------------------------------------------
Hidden layer  0
  Weight: mean=7.1341e-03, std=1.8916e-01, min=-3.3078e-01, max=3.3230e-01
  Bias  : mean=-1.2152e-02, std=1.8760e-01, min=-3.2975e-01, max=3.3306e-01
  Omega scale (exp): mean=1.0000e+00, std=0.0000e+00
------------------------------------------------------------
Hidden layer  1
  Weight: mean=6.2948e-06, std=2.9461e-03, min=-5.1030e-03, max=5.1029e-03
  Bias  : mean=3.2051e-05, std=2.9077e-03, min=-5.0769e-03, max=5.0955e-03
------------------------------------------------------------
Hidden layer  2
  Weight: mean=-2.4295e-05, std=2.9552e-03, min=-5.1031e-03, max=5.1029e-03
  Bias  : mean=-4.6270e-05, std=2.9337e-03, min=-5.0569e-03, max=5.1006e-03
------------------------------------------------------------
Hidden layer  3
  Weight: mean=-1.7248e-05, std=2.9479e-03, min=-5.1030e-03, max=5.1028e-03
  Bias  : mean=1.3535e-04, std=2.7929e-03, min=-4.9978e-03, max=5.0838e-03
------

## Model training without pruning or densification



In [5]:
train(epochs=300, data=mesh, no_surface=10000, no_off_surface=10000, model=model, loss=model_loss, optimizer=optimizer, scene=mesh.scene)

Step 0 | Loss 2.620626211166382
Step 10 | Loss 0.4203188121318817
Step 20 | Loss 0.34520378708839417
Step 30 | Loss 0.2986212372779846
Step 40 | Loss 0.26680684089660645
Step 50 | Loss 0.2447105497121811
Step 60 | Loss 0.22755149006843567
Step 70 | Loss 0.2196338027715683
Step 80 | Loss 0.21388982236385345
Step 90 | Loss 0.2068246454000473
Step 100 | Loss 0.2011401653289795
Step 110 | Loss 0.19748960435390472
Step 120 | Loss 0.1938762664794922
Step 130 | Loss 0.19144386053085327
Step 140 | Loss 0.18753832578659058
Step 150 | Loss 0.18472294509410858
Step 160 | Loss 0.18118081986904144
Step 170 | Loss 0.17994509637355804
Step 180 | Loss 0.17662425339221954
Step 190 | Loss 0.17289495468139648
Step 200 | Loss 0.1723005771636963
Step 210 | Loss 0.1699649840593338
Step 220 | Loss 0.16823521256446838
Step 230 | Loss 0.16422533988952637
Step 240 | Loss 0.16259798407554626
Step 250 | Loss 0.1633356511592865
Step 260 | Loss 0.16044652462005615
Step 270 | Loss 0.1581898033618927
Step 280 | Loss 

#### Model size after pruning

In [5]:
model.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  256 neurons
Hidden layer  2:  256 neurons
Hidden layer  3:  256 neurons
Hidden layer  4:  256 neurons
Final layer    :    1 neurons


In [6]:
marching_cubes.write_obj("armadillo_128_unpruned.obj", model=model, resolution=128, level=0.0)

## Model training with densification

In [6]:
torch.manual_seed(42)
model = si.SIRENSDF()
model_loss = loss.Loss(lambda_surface=150,  lambda_eikonal=0.5, lambda_normal=1.5, lambda_inter=0.5, lambda_sign=1.5, lambda_twd=1e-4, model=model)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
si.weight_stats(model)

Weight statistics per layer:
------------------------------------------------------------
Hidden layer  0
  Weight: mean=7.1341e-03, std=1.8916e-01, min=-3.3078e-01, max=3.3230e-01
  Bias  : mean=-1.2152e-02, std=1.8760e-01, min=-3.2975e-01, max=3.3306e-01
  Omega scale (exp): mean=1.0000e+00, std=0.0000e+00
------------------------------------------------------------
Hidden layer  1
  Weight: mean=6.2948e-06, std=2.9461e-03, min=-5.1030e-03, max=5.1029e-03
  Bias  : mean=3.2051e-05, std=2.9077e-03, min=-5.0769e-03, max=5.0955e-03
------------------------------------------------------------
Hidden layer  2
  Weight: mean=-2.4295e-05, std=2.9552e-03, min=-5.1031e-03, max=5.1029e-03
  Bias  : mean=-4.6270e-05, std=2.9337e-03, min=-5.0569e-03, max=5.1006e-03
------------------------------------------------------------
Hidden layer  3
  Weight: mean=-1.7248e-05, std=2.9479e-03, min=-5.1030e-03, max=5.1028e-03
  Bias  : mean=1.3535e-04, std=2.7929e-03, min=-4.9978e-03, max=5.0838e-03
------

In [7]:
train(epochs=300, data=mesh, no_surface=10000, no_off_surface=10000, model=model, loss=model_loss, optimizer=optimizer, scene=mesh.scene, densification=True)

Step 0 | Loss 2.620621919631958
Step 10 | Loss 0.4200729429721832
Step 20 | Loss 0.34230685234069824
Step 30 | Loss 0.29474976658821106
Step 40 | Loss 0.2648774981498718
Step 50 | Loss 0.24540948867797852
Step 60 | Loss 0.23128141462802887
Step 70 | Loss 0.2236870527267456
Step 80 | Loss 0.21776342391967773
Step 90 | Loss 0.21082279086112976
Step 100 | Loss 0.20712491869926453
Step 110 | Loss 0.20078372955322266
Step 120 | Loss 0.19871747493743896
Step 130 | Loss 0.19262751936912537
Step 140 | Loss 0.19099317491054535
Step 150 | Loss 0.1888267993927002
Step 160 | Loss 0.18471172451972961
Step 170 | Loss 0.18395373225212097
Step 180 | Loss 0.18099358677864075
Step 190 | Loss 0.17798015475273132
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.17725391685962677
Step 210 | Loss 0.22739818692207336
Step 220 | Loss 0.1942848563194275
Step 230 | Loss 0.1777881681919098
Step 240 | Loss 0.16809459030628204
Step 250 | Loss 0.16499541699886322
Step 260 | Loss 0.16117708384990692
St

In [9]:
model.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  256 neurons
Hidden layer  2:  256 neurons
Hidden layer  3:  256 neurons
Hidden layer  4:  256 neurons
Final layer    :    1 neurons


In [10]:
marching_cubes.write_obj("armadillo_128_densified.obj", model=model, resolution=128, level=0.0)

## AIRe: Model training (no densification)

In [8]:
torch.manual_seed(42)
model = si.SIRENSDF()
prune_AIRe = pm.AIRe(model, 0.2, int(size_per_layer/5))
model_loss = loss.Loss(lambda_surface=150,  lambda_eikonal=0.5, lambda_normal=1.5, lambda_inter=0.5, lambda_sign=1.5, lambda_twd=1e-4, model=model, pruning_module=prune_AIRe)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [9]:
train(epochs=300, data=mesh, no_surface=10000, no_off_surface=10000, model=model, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_AIRe)

Step 0 | Loss 2.6333067417144775
Step 10 | Loss 0.4309091866016388
Step 20 | Loss 0.3566991984844208
Step 30 | Loss 0.30522310733795166
Step 40 | Loss 0.2742249667644501
Step 50 | Loss 0.2560141384601593
Step 60 | Loss 0.24088482558727264
Step 70 | Loss 0.23231831192970276
Step 80 | Loss 0.22561952471733093
Step 90 | Loss 0.21744927763938904
Step 100 | Loss 0.21532119810581207
Step 110 | Loss 0.20961979031562805
Step 120 | Loss 0.20754222571849823
Step 130 | Loss 0.20461314916610718
Step 140 | Loss 0.20053623616695404
Step 150 | Loss 0.19792857766151428
Step 160 | Loss 0.19370104372501373
Step 170 | Loss 0.19220033288002014
Step 180 | Loss 0.19026491045951843
Step 190 | Loss 0.18507753312587738
Step 200 | Loss 0.18329250812530518
Step 210 | Loss 0.18150004744529724
Step 220 | Loss 0.179798424243927
Step 230 | Loss 0.17434565722942352
Step 240 | Loss 0.1720939576625824
Pruned 208 neurons.
Step 250 | Loss 0.17180100083351135
Step 260 | Loss 0.24674442410469055
Step 270 | Loss 0.203513383

In [13]:
model.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  204 neurons
Hidden layer  2:  204 neurons
Hidden layer  3:  204 neurons
Hidden layer  4:  204 neurons
Final layer    :    1 neurons


In [14]:
marching_cubes.write_obj("armadillo_128_AIRe.obj", model=model, resolution=128, level=0.0)

## DepGraph: Model training (no densification)

In [15]:
model = si.SIRENSDF()
prune_DepGraph = pm.DepGraph(model, 0.2, int(size_per_layer/5))
model_loss = loss.Loss(lambda_surface=150,  lambda_eikonal=0.5, lambda_normal=1.5, lambda_inter=0.5, lambda_sign=1.5, lambda_twd=1e-4, model=model, pruning_module=prune_DepGraph)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [16]:
train(epochs=750, data=mesh, no_surface=10000, no_off_surface=10000, model=model, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_DepGraph)

Step 0 | Loss 2.4203996658325195
Step 10 | Loss 0.4180848002433777
Step 20 | Loss 0.34326067566871643
Step 30 | Loss 0.29907724261283875
Step 40 | Loss 0.26922959089279175
Step 50 | Loss 0.2459312081336975
Step 60 | Loss 0.23197001218795776
Step 70 | Loss 0.2216319739818573
Step 80 | Loss 0.21357621252536774
Step 90 | Loss 0.20649898052215576
Step 100 | Loss 0.20101670920848846
Step 110 | Loss 0.19671222567558289
Step 120 | Loss 0.19316129386425018
Step 130 | Loss 0.1890491098165512
Step 140 | Loss 0.18466070294380188
Step 150 | Loss 0.18181850016117096
Step 160 | Loss 0.1787284016609192
Step 170 | Loss 0.17843757569789886
Step 180 | Loss 0.1744462251663208
Step 190 | Loss 0.17113937437534332
Step 200 | Loss 0.16995801031589508
Step 210 | Loss 0.168562114238739
Step 220 | Loss 0.1668047457933426
Step 230 | Loss 0.16287264227867126
Step 240 | Loss 0.16139884293079376
Pruned 208 neurons.
Step 250 | Loss 0.15931573510169983
Step 260 | Loss 0.23774509131908417
Step 270 | Loss 0.19956155121

In [17]:
model.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  204 neurons
Hidden layer  2:  204 neurons
Hidden layer  3:  204 neurons
Hidden layer  4:  204 neurons
Final layer    :    1 neurons


In [18]:
marching_cubes.write_obj("armadillo_128_DepGraph.obj", model=model, resolution=128, level=0.0)

## AIRe: Model training with densification

In [10]:
torch.manual_seed(42)
model = si.SIRENSDF()
prune_AIRe = pm.AIRe(model, 0.2, int(size_per_layer/5))
model_loss = loss.Loss(lambda_surface=150,  lambda_eikonal=0.5, lambda_normal=1.5, lambda_inter=0.5, lambda_sign=1.5, lambda_twd=1e-4, model=model, pruning_module=prune_AIRe)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [11]:
train(epochs=300, data=mesh, no_surface=10000, no_off_surface=10000, model=model, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_AIRe, densification=True)

Step 0 | Loss 2.6333014965057373
Step 10 | Loss 0.4318491816520691
Step 20 | Loss 0.3589419424533844
Step 30 | Loss 0.3079119026660919
Step 40 | Loss 0.27767664194107056
Step 50 | Loss 0.25660693645477295
Step 60 | Loss 0.24163983762264252
Step 70 | Loss 0.23376739025115967
Step 80 | Loss 0.2257646918296814
Step 90 | Loss 0.2189064472913742
Step 100 | Loss 0.2127753645181656
Step 110 | Loss 0.2093733251094818
Step 120 | Loss 0.2067776769399643
Step 130 | Loss 0.20303940773010254
Step 140 | Loss 0.20202705264091492
Step 150 | Loss 0.19767804443836212
Step 160 | Loss 0.19352641701698303
Step 170 | Loss 0.1925715208053589
Step 180 | Loss 0.18749940395355225
Step 190 | Loss 0.18680931627750397
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.1851765215396881
Step 210 | Loss 0.2406916469335556
Step 220 | Loss 0.2091096192598343
Step 230 | Loss 0.19311442971229553
Step 240 | Loss 0.183036208152771
Pruned 208 neurons.
Step 250 | Loss 0.17907455563545227
Step 260 | Loss 0.2323198

In [21]:
model.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  204 neurons
Hidden layer  2:  204 neurons
Hidden layer  3:  204 neurons
Hidden layer  4:  204 neurons
Final layer    :    1 neurons


In [22]:
marching_cubes.write_obj("armadillo_128_AIRe_densified.obj", model=model, resolution=128, level=0.0)

## DepGraph: Model training with densification

In [23]:
model = si.SIRENSDF()
prune_DepGraph = pm.DepGraph(model, 0.2, int(size_per_layer/5))
model_loss = loss.Loss(lambda_surface=150,  lambda_eikonal=0.5, lambda_normal=1.5, lambda_inter=0.5, lambda_sign=1.5, lambda_twd=1e-4, model=model, pruning_module=prune_DepGraph)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [24]:
train(epochs=750, data=mesh, no_surface=10000, no_off_surface=10000, model=model, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_DepGraph, densification=True)

Step 0 | Loss 2.4881484508514404
Step 10 | Loss 0.42603787779808044
Step 20 | Loss 0.34035348892211914
Step 30 | Loss 0.29318034648895264
Step 40 | Loss 0.26054999232292175
Step 50 | Loss 0.2407781481742859
Step 60 | Loss 0.22648334503173828
Step 70 | Loss 0.2188604772090912
Step 80 | Loss 0.21292035281658173
Step 90 | Loss 0.20777815580368042
Step 100 | Loss 0.20187415182590485
Step 110 | Loss 0.19904662668704987
Step 120 | Loss 0.1920287162065506
Step 130 | Loss 0.18880000710487366
Step 140 | Loss 0.18591439723968506
Step 150 | Loss 0.1837618350982666
Step 160 | Loss 0.1804526001214981
Step 170 | Loss 0.17698359489440918
Step 180 | Loss 0.1743568778038025
Step 190 | Loss 0.1711864024400711
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.17030180990695953
Step 210 | Loss 0.234788715839386
Step 220 | Loss 0.20083975791931152
Step 230 | Loss 0.1817893534898758
Step 240 | Loss 0.17485609650611877
Pruned 208 neurons.
Step 250 | Loss 0.16949139535427094
Step 260 | Loss 0.219

In [25]:
model.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  204 neurons
Hidden layer  2:  204 neurons
Hidden layer  3:  204 neurons
Hidden layer  4:  204 neurons
Final layer    :    1 neurons


In [26]:
marching_cubes.write_obj("armadillo_128_DepGraph_densified.obj", model=model, resolution=128, level=0.0)

In [27]:
from src.mesh_extraction import marching_cubes_test 
marching_cubes_test.write_obj("armadillo_128_gt.obj", mesh.scene, 128, 0.0)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Sample a few surface points and check their SDF values
test_points = mesh.vertices[:10]  # First 10 points
test_tensor = torch.from_numpy(test_points).float().to(device)
with torch.no_grad():
    sdf_values = model(test_tensor)
print("SDF values for surface points:")
print(sdf_values)
print("Mean absolute SDF:", torch.abs(sdf_values).mean().item())

SDF values for surface points:
tensor([[-0.0152],
        [-0.0121],
        [-0.0171],
        [-0.0210],
        [-0.0124],
        [-0.0125],
        [-0.0135],
        [-0.0142],
        [-0.0179],
        [-0.0182]], device='cuda:0')
Mean absolute SDF: 0.015407375991344452


In [29]:
# import src.mesh_extraction.marching_cubes_test as marching_cubes_test
# marching_cubes_test.write_obj("armadillo_mesh_ground_truth_128.obj", scene=mesh.scene, resolution=128, level=0.0)


In [30]:
# Make batched point
test_point = torch.from_numpy(np.array([[-1, -1, -1]])).float().to(device)

# Compute SIREN model prediction
sdf_pred = model(test_point)
print("SIREN prediction:", sdf_pred)

# Compute Open3D signed distance
distance = mesh.scene.compute_signed_distance(
    o3d.core.Tensor(test_point.cpu().numpy(), dtype=o3d.core.Dtype.Float32)
)
print("Open3D SDF:", distance.numpy())

SIREN prediction: tensor([[0.0355]], device='cuda:0', grad_fn=<AddmmBackward0>)
Open3D SDF: [0.96991843]
